In [ ]:
import os
import sys
import numpy as np

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
from sklearn.metrics import precision_score, recall_score, f1_score
from scipy.stats import chi2_contingency


In [ ]:
thresholds = ['A','I','N'] #['0.1','0.5','0.8', 
hardware = ["kyiv", "brisbane", "sherbrooke"]
mutant_types = ["equiv", "normal", "balanced"]
metrics = ['C', 'H', 'J', 'T', 'F', 'E']
metric_names=('Chisquare', 'Hellinger', 'Jensen-shannon', 'Trace', 'Fidelity', 'Expectation Values')
output_type = {'ae': 'Dominant', 'qpeexact': 'Dominant', 'vqe': 'Dominant', 'qft': 'Diverse', 'qftentangled': 'Diverse', 'wstate': 'Diverse'}


# Merge and save DFs for equiv, normal and balanced

In [ ]:
def read_and_merge_csv_files(folder_path):
    # List to hold individual DataFrames
    dataframes = []

    # Loop through all files in the folder
    for filename in os.listdir(folder_path):
        if filename.endswith('.csv'):
            file_path = os.path.join(folder_path, filename)
            # Read the CSV file into a DataFrame
            df = pd.read_csv(file_path)
            # Append the DataFrame to the list
            dataframes.append(df)

    # Concatenate all DataFrames in the list into a single DataFrame
    merged_df = pd.concat(dataframes, ignore_index=True)

    return merged_df

# Function to split the name column and create new columns
def split_name_column(name):
    name = name.replace('.qasm', '')
    parts = name.split('_')
    position = int(parts[5].replace('P', ''))
    qubit = parts[6].replace('Q', '')
    
    if len(parts) > 7: 
        parameters = parts[7].strip('[]') 
    else: 
        parameters = None

    return parts[0], parts[1], parts[3], parts[4], position, qubit, parameters

def get_gate_type(gate):
    single_qubit_gates = ["x", "h", "p", "t", "s", "z", "y", "id", "rx", "ry", "rz", "sx", "u", "u1", "u2", "u3"]
    multi_qubit_gates = ["swap", "rzz", "rxx", "cx", "cz", "cp", "ccx", "cswap", "ch"]
    if gate in single_qubit_gates:
        return 'Single_qubit'
    elif gate in multi_qubit_gates:
        return 'Multi_qubit'
    else:
        return 'Gate_not_supported'
    
# Function to categorize position based on percentage
def categorize_position(percentage):
    if percentage <= 20:
        return 'beginning'
    elif percentage <= 40:
        return 'pre_middle'
    elif percentage <= 60:
        return 'middle'
    elif percentage <= 80:
        return 'post_middle'
    else:
        return 'end'

In [ ]:
def get_dataframe(model, mutant, threshold, df_char):
    # Get all the results in a df
    folder_path = f'./results_{model}/results_{mutant}_{threshold}'
    df = read_and_merge_csv_files(folder_path)
    
    # Create a column to categorize the input type
    df[['Input_type']] = df['Input'].apply(lambda x: pd.Series(x.split('_')[0]))
    
    # Apply the function to the name column and create new columns
    df[['Algorithm', 'Qubits_number', 'Operator', 'Gate', 'Position', 'Qubits', 'Params']] = df['Name'].apply(lambda x: pd.Series(split_name_column(x)))
    
    # Create a column to categorize the gate type
    df[['Gate_type']] = df['Gate'].apply(lambda x: pd.Series(get_gate_type(x)))
    
    # Calculate position percentage and categorize it
    df['max_position'] = df.groupby(['Algorithm', 'Qubits_number'])['Position'].transform('max')
    df['position_percentage'] = (df['Position'] / df['max_position']) * 100
    df['Relative_position'] = df['position_percentage'].apply(categorize_position)
    
    # Drop the intermediate columns if needed
    df = df.drop(columns=['max_position', 'position_percentage'])
    df = df.drop(columns=['Name'])
    
    # Merge df and df_charac on 'qubits'/'Qubits' and 'algo'/'Algorithm'
    merged_df = pd.merge(df_char, df, left_on=['qubits', 'algo'], right_on=['Qubits_number', 'Algorithm'], how='right')
    merged_df = merged_df.drop(columns=['qubits'])  # or 'Qubits_number' if you prefer to keep the original name
    df = merged_df.drop(columns=['algo'])  # or 'Qubits_number' if you prefer to keep the original name
    
    # First, replace the 'Algorithm' column values with the dictionary mapping
    df['Output_type'] = df['Algorithm'].map(output_type)
    
    csv_path = f'results/dataframes/{model}_{mutant}_{threshold}.csv'
    df.to_csv(csv_path, mode='w', header=True, index=False)

    return df

In [ ]:
def get_balanced_df(df_equiv, df_normal, csv_path, df_char):
    
    n_rows = min(len(df_equiv), len(df_normal))
    
    # Randomly sample from each DataFrame
    df_equiv_sampled = df_equiv.sample(n=n_rows, random_state=42)
    df_normal_sampled = df_normal.sample(n=n_rows, random_state=42)
    
    # Combine and shuffle the rows from both types
    balanced_df = pd.concat([df_equiv_sampled, df_normal_sampled]).sample(frac=1, random_state=1).reset_index(drop=True)
    
    # First, replace the 'Algorithm' column values with the dictionary mapping
    balanced_df['Output_type'] = balanced_df['Algorithm'].map(output_type)

    balanced_df.to_csv(csv_path, mode='w', header=True, index=False)
    
    return balanced_df

In [ ]:
output_folder = 'results/dataframes'
os.makedirs(output_folder, exist_ok=True)

# Load the .xlsx file
xlsx_path = 'data/origin_qc/programs_characteristics.xlsx'
df_charac = pd.read_excel(xlsx_path, usecols=[0, 2, 3, 5, 6, 7])
df_charac['algo'] = df_charac.iloc[:, 0].str.split('_').str[0]  # Extract the algorithm name (first part)
df_charac['qubits'] = df_charac['qubits'].astype(str)
df_charac = df_charac.drop(columns=[df_charac.columns[0]])

for threshold in thresholds:
    for hw in hardware:
        
        # Load 'equiv' and 'normal' DataFrames once
        df_equiv = get_dataframe(hw, "equiv", threshold, df_charac)
        df_normal = get_dataframe(hw, "normal", threshold, df_charac)
        csv_balanced_path = f'results/dataframes/{hw}_balanced_{threshold}.csv'
        df_balanced = get_balanced_df(df_equiv, df_normal, csv_balanced_path, df_charac)

In [ ]:
output_folder = 'results'
os.makedirs(output_folder, exist_ok=True)
metrics_names = {'C':'chisquare', 'H':'hellinger', 'J':'jensenshannon', 'T':'trace', 'F':'fidelity', 'E':'expectation'}

for mutant_type in mutant_types:
    dataframes = []
    for threshold in thresholds:
        for hw in hardware:
            csv_path = f'results/dataframes/{hw}_{mutant_type}_{threshold}.csv'
            df = pd.read_csv(csv_path)
            df['hardware'] = hw
            df['threshold'] = threshold
            dataframes.append(df)
    
    complete_df = pd.concat(dataframes, ignore_index=True)
    selected_columns = complete_df[['gates', 'depth', 'singlequbit_gates', 'multiqubit_gates', 'Input', 'Input_type', 'Algorithm', 'Qubits_number', 'Operator', 'Gate', 'Position', 'Qubits', 'Gate_type', 'Relative_position', 'Output_type', 'hardware', 'threshold']]
        
    new_rows = []

    for metric, metric_name in metrics_names.items():
        # Extract true and predicted labels for the current metric
        true_labels = complete_df[f'Killed_I{metric}']
        predicted_labels = complete_df[f'Killed_N{metric}']
        metric_df = selected_columns.copy()
        metric_df['metric'] = metric  
        
        metric_df['true_label'] = true_labels
        metric_df['predicted_label'] = predicted_labels
        metric_df['distance'] = complete_df[f'Noisy_{metric_name}']

        metric_df['correctness'] = (true_labels == predicted_labels)  
        new_rows.append(metric_df)
    
    metric_df = pd.concat(new_rows, ignore_index=True)
    metric_df.reset_index(drop=True, inplace=True)

    output_path = os.path.join(output_folder, f'results_{mutant_type}_selected.csv')
    metric_df.to_csv(output_path, index=False)   


# F1, Precision and recall

In [ ]:
# TODO: metrics into dic

In [ ]:
def get_scores(df):
    results = []
    
    for metric in metrics:
        
        true_labels = df[f'Killed_I{metric}']
        predicted_labels = df[f'Killed_N{metric}']
        
        # Calculate precision, recall, and F1 score for the pair
        precision = precision_score(true_labels, predicted_labels, zero_division=0)
        recall = recall_score(true_labels, predicted_labels, zero_division=0)
        f1 = f1_score(true_labels, predicted_labels, zero_division=0)
        results.append((precision, recall, f1))
    
    scores_df = pd.DataFrame(results, columns=['Precision', 'Recall', 'F1 Score'], 
                             index=[f'Metric {metric}' for metric in metrics])
    
    return scores_df


In [ ]:
output_folder = 'results/scores'
os.makedirs(output_folder, exist_ok=True)

scores = ["Precision", "Recall", "F1 Score"]

for mutant_type in mutant_types:
    # Initialize dictionaries to store score DataFrames with (metric, threshold) index
    dic_scores = {
        score: pd.DataFrame(columns=hardware, index=pd.MultiIndex.from_product(
            [[f'Metric {metric}' for metric in metrics], thresholds], names=["Metric", "Threshold"]
        ))
        for score in scores
    }
    
    for threshold in thresholds:
        for hw in hardware:
            csv_path = f'results/dataframes/{hw}_{mutant_type}_{threshold}.csv'
            df = pd.read_csv(csv_path)
            scores_df = get_scores(df)
            
            for score in scores:
                for metric in scores_df.index:
                    dic_scores[score].at[(metric, threshold), hw] = scores_df.at[metric, score]

    # Save each score DataFrame as a CSV
    for score, df_score in dic_scores.items():
        output_path = os.path.join(output_folder, f'{mutant_type}_{score}.csv')
        df_score.to_csv(output_path)
        print(f"Saved {output_path}")


# Confusion matrices


In [ ]:
metrics = ['H', 'J', 'T', 'F', 'E'] #['C', 'H', 'J', 'T', 'F', 'E']
metric_names= ('Hellinger', 'Jensen-shannon', 'Trace', 'Fidelity', 'Expectation Values') #('Chisquare', 'Hellinger', 'Jensen-shannon', 'Trace', 'Fidelity', 'Expectation Values')

In [ ]:
def confusion_matrix(df, col1, col2):
    # Create a confusion matrix DataFrame
    conf_matrix = pd.DataFrame(index=['True', 'False'], columns=['True', 'False'])
    
    # Calculate the count of each pair
    true_true = ((df[col1] == True) & (df[col2] == True)).sum()
    false_false = ((df[col1] == False) & (df[col2] == False)).sum()
    true_false = ((df[col1] == True) & (df[col2] == False)).sum()
    false_true = ((df[col1] == False) & (df[col2] == True)).sum()
    
    # Total number of rows
    total = len(df)
    
    # Calculate percentages
    conf_matrix.loc['True', 'True'] = (true_true / total) * 100
    conf_matrix.loc['False', 'False'] = (false_false / total) * 100
    conf_matrix.loc['True', 'False'] = (true_false / total) * 100
    conf_matrix.loc['False', 'True'] = (false_true / total) * 100
    
    # Ensure all values are numeric and handle any potential issues
    conf_matrix = conf_matrix.apply(pd.to_numeric, errors='coerce')  # Convert to numeric, coerce errors to NaN
    conf_matrix.fillna(0, inplace=True)  # Replace NaNs with 0 if there are any

    return conf_matrix

In [ ]:
# Define a function to create a heatmap with annotations
def create_heatmap(fig, data, row, col, showscale):
    fig.add_trace(
        go.Heatmap(
            z=data,
            text=data,  # Use the same data for annotations
            colorscale= [[0.0, '#eff3ff'], [0.05, '#9ecae1'],[0.1, '#6baed6'], [0.8, '#3182bd'], [1, '#08519c']],
            colorbar=dict(title='Scale'),
            zmin=0, zmax=100,
            showscale=showscale,
            texttemplate='%{text:.2f}',  # Format the text annotations
            textfont=dict(size=18)
        ),
        row=row, col=col
    )
    
    fig.update_xaxes(tickvals=[0, 1], ticktext=['Killed', 'Survived'], row=row, col=col)
    fig.update_yaxes(tickvals=[0, 1], ticktext=['Killed', 'Survived'], row=row, col=col)
    

In [ ]:
def print_confusion_matrices(hw, threshold, mutant):
    # Create subplots with titles
    fig = make_subplots(
        rows=1, cols=6,
        subplot_titles=metric_names,
        x_title='Noisy', y_title='Ideal', horizontal_spacing=0.05
    )
    
    csv_path = f'results/dataframes/{hw}_{mutant_type}_{threshold}.csv'
    df_confusion = pd.read_csv(csv_path)
    
    for j, metric in enumerate(metrics):
        matrix = confusion_matrix(df_confusion, 'Killed_I' + metric, 'Killed_N' + metric)
        # Add heatmaps to subplots
        create_heatmap(fig, matrix, row=1, col=j+1, showscale=True)
    
    # Update layout and save as image
    fig.update_layout(
        title_text=f'Overall confusion matrix for {mutant} mutants on {hw} with a threshold of {threshold}',
        height=400,
        width=2000,
        showlegend=False
    )
    
    #fig.show()
    fig.write_image(f"results/no_chi_square/confusion_matrices/{hw}_{mutant}_{threshold}.png")

In [ ]:
output_folder = 'results/no_chi_square/confusion_matrices/'
os.makedirs(output_folder, exist_ok=True)

for threshold in thresholds:
    for hw in hardware:
        for mutant_type in mutant_types:
            print_confusion_matrices(hw, threshold, mutant_type)

In [ ]:
# MERGE HW

In [ ]:
def print_confusion_matrices(threshold, mutant):
    # Create subplots with titles
    fig = make_subplots(
        rows=1, cols=6,
        subplot_titles=metric_names,
        x_title='Noisy', y_title='Ideal', horizontal_spacing=0.05
    )
    
    dataframes = []
    for hw in hardware:
        csv_path = f'results/dataframes/{hw}_{mutant_type}_{threshold}.csv'
        dataframes.append(pd.read_csv(csv_path))
    
    df_confusion = pd.concat(dataframes, ignore_index=True)
    
    for j, metric in enumerate(metrics):
        matrix = confusion_matrix(df_confusion, 'Killed_I' + metric, 'Killed_N' + metric)
        # Add heatmaps to subplots
        print(matrix)
        create_heatmap(fig, matrix, row=1, col=j+1, showscale=True)
    
    # Update layout and save as image
    fig.update_layout(
        title_text=f'Overall confusion matrix for {mutant} mutants with a threshold of {threshold}',
        height=400,
        width=2000,
        showlegend=False
    )
    
    #fig.show()
    fig.write_image(f"results/no_chi_square/confusion_matrices/global/{mutant}_{threshold}.png")

In [ ]:
output_folder = 'results/no_chi_square/confusion_matrices/global/'
os.makedirs(output_folder, exist_ok=True)

for threshold in thresholds:
    for mutant_type in mutant_types:
        print_confusion_matrices(threshold, mutant_type)

# Helper functions

In [ ]:
# Helper function to calculate F1 scores
def calculate_scores(df, categories, metrics, name):
    scores = {}
    for metric in metrics:
        for cat in categories:
            filtered_df = df.loc[df[name] == cat]
            if not filtered_df.empty:
                true_labels = filtered_df[f'Killed_I{metric}']
                predicted_labels = filtered_df[f'Killed_N{metric}']
                scores[cat, metric] = f1_score(true_labels, predicted_labels, zero_division=0.0)
            else:
                scores[cat, metric] = None  # Default for empty cases
    return scores
    
    
# Helper function to setup layout and save image
def setup_layout_and_save(fig, title, folder_name, file_name, yaxis_range=None):
    fig.update_layout(
        title_text=title,
        height=400,
        width=2000,
        showlegend=True,
        yaxis_range=yaxis_range  # Set y-axis range if provided
    )
    os.makedirs(folder_name, exist_ok=True)
    fig.write_image(f"{folder_name}/{file_name}.png")# engine='orca')


In [ ]:
def line_interpolation(scores):
    # Assuming scores is a dictionary structured as {(category, metric): score}
    scores_df = pd.DataFrame.from_dict(scores, orient='index', columns=['F1_Score'])
    scores_df.reset_index(inplace=True)
    scores_df.columns = ['Category_Metric', 'F1_Score']
            
    # Use interpolation to fill missing values
    scores_df['F1_Score'] = scores_df['F1_Score'].interpolate(method='linear')
            
    # Convert back to a dictionary if needed
    scores_interpolated = {(row['Category_Metric']): row['F1_Score'] for _, row in scores_df.iterrows()}
        
    return scores_interpolated     

# Line Graphs

In [ ]:
def print_line_interpolation(df_threshold, name, categories, metrics, metric_names):
    
    fig = go.Figure()
    color_scale = px.colors.qualitative.Bold
    scores = {}
    
    # Loop over the groups to add each one to the figure
    for i, metric in enumerate(metrics):
        
        #threshold = metrics[metric]
        #df_threshold = get_dataframe(threshold)
        need_interpolation = False
        
        for cat in categories:
            
            if name == 'Qubits_number':
                filtered_df = df_threshold.loc[df[name] == cat]
            else:
                filtered_df = df_threshold.loc[df[name] == int(cat)]
            
            if filtered_df.empty:
                scores[cat, metric] = None  # Or you can assign NaN or a default value
                need_interpolation = True
            else:
                true_labels = filtered_df[f'Killed_I{metric}']
                predicted_labels = filtered_df[f'Killed_N{metric}']
                # Calculate F1 score for the pair
                scores[cat, metric] = f1_score(true_labels, predicted_labels, zero_division=0)
    
        if need_interpolation:
            scores = line_interpolation(scores)
            
    for i, metric in enumerate(metrics):
        y_values = [scores[(cat, metric)] for cat in categories]
        fig.add_trace(go.Scatter(
            name=metric_names[i],  # Name of the operator
            x=categories,  # Metrics on x-axis
            y=y_values,  # F1 scores for the current operator
            hoverinfo='y',  # Show F1 score on hover
            marker=dict(color=color_scale[i % len(color_scale)])  # Assign a color from the color scale
        ))
    
    setup_layout_and_save(fig, name, f'results/no_chi_square/line_charts/{name}', file_name, yaxis_range=[0, 1])


In [ ]:
def print_line_chart(df_threshold, name, categories, file_name):
    
    fig = go.Figure()
    color_scale = px.colors.qualitative.Bold
    scores = calculate_scores(df_threshold, categories, metrics, name)
    
    for i, metric in enumerate(metrics):
        y_values = [scores[(cat, metric)] for cat in categories]
        fig.add_trace(go.Scatter(
            name=metric_names[i],  # Name of the operator
            x=categories,  # Metrics on x-axis
            y=y_values,  # F1 scores for the current operator
            hoverinfo='y',  # Show F1 score on hover
            marker=dict(color=color_scale[i % len(color_scale)])  # Assign a color from the color scale
        ))

    setup_layout_and_save(fig, name, f'results/no_chi_square/line_charts/{name}', file_name, yaxis_range=[0, 1])


In [ ]:
columns = ['Qubits_number', 'gates', 'depth', 'singlequbit_gates', 'multiqubit_gates'] 

for threshold in thresholds:
    for hw in hardware:
        for mutant_type in mutant_types:
            csv_path = f'results/dataframes/{hw}_{mutant_type}_{threshold}.csv'
            df = pd.read_csv(csv_path)
            for cat in columns: 
                min_val = int(df[cat].min())
                max_val = int(df[cat].max())
                cat_range = list(map(int, range(min_val, max_val + 1)))  
                file_name = f'{hw}_{mutant_type}_{threshold}'
                print_line_chart(df, cat, cat_range, file_name)
                print(f'{cat}: [{min_val}, {max_val}]')

# Bar graphs

In [ ]:
def print_grouped_bar_chart(df_threshold, name, categories, file_name):
    fig = go.Figure()
    color_scale = px.colors.qualitative.Bold
    scores = calculate_scores(df_threshold, categories, metrics, name)
    
    # Add bars for each category
    for i, cat in enumerate(categories):
        y_values = [scores[(cat, metric)] for metric in metrics]
        fig.add_trace(go.Bar(
            name=cat,
            x=metric_names,
            y=y_values,
            hoverinfo='y',
            marker=dict(color=color_scale[i % len(color_scale)])
        ))
    
    setup_layout_and_save(fig, name, f'results/no_chi_square/bar_charts/{name}', file_name)

In [ ]:
table_data = {
    "Output_type": ["Dominant", "Diverse"],
    "Input_type": ["PureState", "Quratest"],
    "Operator": ["Add", "Remove", "Replace"],
    "Gate_type": ["Single_qubit", "Multi_qubit"],
    "Relative_position": ["beginning", "pre_middle", "middle", "post_middle", "end"]
}

for threshold in thresholds:
    for hw in hardware:
        for mutant_type in mutant_types:
            csv_path = f'results/dataframes/{hw}_{mutant_type}_{threshold}.csv'
            df = pd.read_csv(csv_path)
            for key, categories in table_data.items():
                file_name = f'{hw}_{mutant_type}_{threshold}'
                print_grouped_bar_chart(df, key, categories, file_name)

# Bar chart F1 Score

In [ ]:
def get_summary(scores):
     # Find highest and lowest thresholds for each metric
    summary_data = []

    for mutant_type in set(mutant_type for (mutant_type, _) in scores.keys()):
            for metric in metrics:
                metric_values = {f'{metric}_{threshold}': [] for threshold in thresholds}
    
                # Collect all values for the metric thresholds and mutant type
                for (mt, metric_threshold), value in scores.items():
                    if mt == mutant_type and metric_threshold.startswith(metric):
                        metric_values[metric_threshold].append(value)
    
                # Calculate the max and min values across thresholds for this mutant type
                max_threshold = max(metric_values, key=lambda k: max(metric_values[k], default=float('-inf')))
                min_threshold = min(metric_values, key=lambda k: min(metric_values[k], default=float('inf')))
                summary_data.append({
                    "Mutant Type": mutant_type,
                    "Metric": metric,
                    "Max Threshold": max_threshold,
                    "Max Value": max(metric_values[max_threshold], default=None),
                    "Min Threshold": min_threshold,
                    "Min Value": min(metric_values[min_threshold], default=None)
                })
    
    # Convert summary data to DataFrame
    summary_df = pd.DataFrame(summary_data)
    return summary_df

In [ ]:
def get_grouped_bar_chart(df, noise_model):
    fig = go.Figure()
    color_scale = px.colors.qualitative.Bold
    
    metric_thresholds = []
    for metric in metrics:
        for threshold in thresholds:
            metric_threshold = f'{metric}_{threshold}'
            metric_thresholds.append(metric_threshold)
    
    scores = {(row["mutant_type"], f"{row['Metric']}_{row['Threshold']}"): row[noise_model] for _, row in df.iterrows()}
    
    # Add bars for each category
    for i, mutant_type in enumerate(mutant_types):
        y_values = [scores[(mutant_type, metric_threshold)] for metric_threshold in metric_thresholds]
        fig.add_trace(go.Bar(
            name=mutant_type, # mutant type
            x=metric_thresholds, # metric_thresholds
            y=y_values, # F1 scores
            hoverinfo='y',
            marker=dict(color=color_scale[i % len(color_scale)])
        ))
    
    return fig, get_summary(scores)

In [ ]:
metric_summary = []

for score in ['Precision', 'Recall', 'F1 Score']:
    dfs = []
    for mutant_type in mutant_types:
        # Save each score DataFrame as a CSV
        csv_path = f'results/scores/{mutant_type}_{score}.csv'
        mutant_type_df = pd.read_csv(csv_path)
        mutant_type_df['mutant_type'] = mutant_type  
        dfs.append(mutant_type_df)
    
    df = pd.concat(dfs, ignore_index=True)
    df["Metric"] = df["Metric"].str.replace("Metric ", "")
    
    fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.1)
    
    for i, hw in enumerate(hardware):
        # Get the grouped bar chart figure
        grouped_bar_chart, summary = get_grouped_bar_chart(df, hw)
        summary['Noise'] = hw
        summary['Score'] = score
        metric_summary.append(summary)
    
        # Add each trace from the grouped bar chart to the subplot
        for trace in grouped_bar_chart.data:
            fig.add_trace(trace, row=i+1, col=1)
        
    fig.update_layout(
        annotations=[
            dict(
                text="Kyiv",  # Title for the first plot
                x=-0.03, y=0.92,  # Position of the title (x, y)
                xref="paper", yref="paper",  # Reference to paper coordinates
                showarrow=False,
                font=dict(size=12),
                textangle=-90  # Rotate text vertically
            ),
            dict(
                text="Brisbane",  # Title for the second plot
                x=-0.03, y=0.5,
                xref="paper", yref="paper",
                showarrow=False,
                font=dict(size=12),
                textangle=-90  # Rotate text vertically
            ),
            dict(
                text="Sherbrooke",  # Title for the third plot
                x=-0.03, y=-0.05,
                xref="paper", yref="paper",
                showarrow=False,
                font=dict(size=12),
                textangle=-90  # Rotate text vertically
            ),
        ]
    )
    setup_layout_and_save(fig, f"Comparison of {score} score per Noise Model", f'results_test/bar_charts_scores/', f"compare_{score}")
    fig.show()
    

summary_df = pd.concat(metric_summary, ignore_index=True)
csv_path = f'./results_test/best_scores.csv'
summary_df.to_csv(csv_path, mode='w', header=True, index=False)


In [ ]:
  print(summary_df.head())

In [ ]:
df = summary_df[['Mutant Type','Metric','Max Threshold', 'Max Value', 'Min Threshold', 'Min Value','Score','Noise']]

# Reshape the DataFrame
reshaped_df = df.pivot(
    index=["Metric", "Mutant Type", "Score"],
    columns="Noise",
    values=["Max Threshold",'Max Value', 'Min Threshold', 'Min Value']
)

# Flatten the multi-index columns for readability
reshaped_df.columns = ['_'.join(col).strip() for col in reshaped_df.columns.values]
reshaped_df = reshaped_df.reset_index()

# Print the reshaped DataFrame
print(reshaped_df)
csv_path = f'./results_test/ordered_scores.csv'
reshaped_df.to_csv(csv_path, mode='w', header=True, index=False)

In [ ]:
def get_grouped_bar_chart(df, noise_model):
    fig = go.Figure()
    color_scale = px.colors.qualitative.Bold
    
    best_metric_threshold = {'C':'0.1', 'H':'0.1', 'J':'0.1', 'T':'I', 'F':'I', 'E':'I'}
    metric_thresholds = []
    for k,v in best_metric_threshold.items():
        metric_threshold = f'{k}_{v}'
        metric_thresholds.append(metric_threshold)
    
    scores = {(row["mutant_type"], f"{row['Metric']}_{row['Threshold']}"): row[noise_model] for _, row in df.iterrows()}
    
    # Add bars for each category
    for i, mutant_type in enumerate(mutant_types):
        y_values = [scores[(mutant_type, metric_threshold)] for metric_threshold in metric_thresholds]
        fig.add_trace(go.Bar(
            name=mutant_type, # mutant type
            x=metric_thresholds, # metric_thresholds
            y=y_values, # F1 scores
            hoverinfo='y',
            marker=dict(color=color_scale[i % len(color_scale)])
        ))
    
    return fig

In [ ]:
for score in ['Precision', 'Recall', 'F1 Score']:
    dfs = []
    for mutant_type in mutant_types:
        # Save each score DataFrame as a CSV
        csv_path = f'results/scores/{mutant_type}_{score}.csv'
        mutant_type_df = pd.read_csv(csv_path)
        mutant_type_df['mutant_type'] = mutant_type  
        dfs.append(mutant_type_df)
    
    df = pd.concat(dfs, ignore_index=True)
    df["Metric"] = df["Metric"].str.replace("Metric ", "")
    
    fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.1)
    
    for i, hw in enumerate(hardware):
        # Get the grouped bar chart figure
        grouped_bar_chart = get_grouped_bar_chart(df, hw)
       
    
        # Add each trace from the grouped bar chart to the subplot
        for trace in grouped_bar_chart.data:
            fig.add_trace(trace, row=i+1, col=1)
        
    fig.update_layout(
        annotations=[
            dict(
                text="Kyiv",  # Title for the first plot
                x=-0.03, y=0.92,  # Position of the title (x, y)
                xref="paper", yref="paper",  # Reference to paper coordinates
                showarrow=False,
                font=dict(size=12),
                textangle=-90  # Rotate text vertically
            ),
            dict(
                text="Brisbane",  # Title for the second plot
                x=-0.03, y=0.5,
                xref="paper", yref="paper",
                showarrow=False,
                font=dict(size=12),
                textangle=-90  # Rotate text vertically
            ),
            dict(
                text="Sherbrooke",  # Title for the third plot
                x=-0.03, y=-0.05,
                xref="paper", yref="paper",
                showarrow=False,
                font=dict(size=12),
                textangle=-90  # Rotate text vertically
            ),
        ]
    )
    setup_layout_and_save(fig, f"Comparison of {score} score per Noise Model", f'results_test/bar_charts_scores/test/', f"compare_{score}")
    fig.show()